# Phase 2.4 — Analyse formelle

**Projet** : COVID-19 et scolarisation au Sénégal — Panel ménages 2021-2022
**Notebook** : `python/02_analysis.ipynb`

Ce notebook produit les **livrables analytiques finaux** :
- Estimations économétriques avec écarts-types correctement spécifiés (cluster ménage)
- Tableaux d'agrégats prêts à charger dans Power BI
- Figures publication-ready exportées en PNG
- Synthèse des trois insights à mettre en avant

Méthodologie : **modèle linéaire de probabilité (LPM)** avec contrôle d'âge en effets fixes pour les comparaisons inter-vagues, et SE clusterisés sur `hh_id` pour tenir compte de la corrélation intra-ménage. Le LPM est préféré au logit pour la lisibilité des coefficients en points de pourcentage, et reste valide pour des taux loin de 0 et 1 (cas ici, taux entre 70 et 95 %).

## Setup

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.filterwarnings('ignore', category=FutureWarning)

# Affichage
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.3f}'.format)
sns.set_theme(style='whitegrid', context='notebook', palette='deep')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 200
plt.rcParams['savefig.bbox'] = 'tight'

# Chemins
ROOT = Path.cwd().parents[0] if Path.cwd().name == 'python' else Path.cwd()
PROC = ROOT / 'data' / 'processed'
FIGS = ROOT / 'docs' / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)
print('ROOT  :', ROOT)
print('PROC  :', PROC)
print('FIGS  :', FIGS)

In [ ]:
# Chargement
enfants = pd.read_parquet(PROC / 'enfants_panel.parquet')
menages = pd.read_parquet(PROC / 'menages_panel.parquet')

# Échantillon analytique principal : enfants observés aux 2 vagues
panel = enfants[enfants['statut_enfant'] == 'panel_present'].copy()

# Types confort
panel['scolarise_bin'] = (panel['scolarise'] == 'Oui').astype(float)
panel.loc[panel['scolarise'].isna(), 'scolarise_bin'] = np.nan
panel['femme'] = (panel['sexe_enfant'] == 'Femme').astype(float)
panel.loc[panel['sexe_enfant'].isna(), 'femme'] = np.nan
panel['wave_2022'] = (panel['wave'] == 2022).astype(int)

print(f"Échantillon panel : {len(panel)} obs (enfant-vague), {panel['hh_id'].nunique()} ménages, {panel[['hh_id','enfant_slot']].drop_duplicates().shape[0]} enfants uniques")

## 1. Question 1 — La baisse 2021→2022 est-elle réelle, ajustée pour l'âge ?

**Spécification** :
$$ Y_{iv} = \alpha + \beta \cdot \mathbb{1}[v=2022] + \sum_a \gamma_a \cdot \mathbb{1}[\text{âge}_{iv}=a] + \varepsilon_{iv} $$

où $Y_{iv}$ est la probabilité de scolarisation de l'enfant $i$ à la vague $v$, et $\beta$ capture l'écart entre vagues **à composition d'âge constante**. SE clusterisés sur `hh_id`.

In [ ]:
# Échantillon : enfants panel avec âge et scolarisation observés aux 2 vagues
ech1 = panel.dropna(subset=['scolarise_bin', 'age_enfant']).copy()
ech1['age_int'] = ech1['age_enfant'].astype(int)
# Restreindre âge raisonnable
ech1 = ech1[ech1['age_int'].between(3, 22)]
print(f"Échantillon Q1 : {len(ech1)} obs")

# Modèle 1a : brut (sans contrôle d'âge)
m1a = smf.ols('scolarise_bin ~ wave_2022', data=ech1).fit(
    cov_type='cluster', cov_kwds={'groups': ech1['hh_id']})

# Modèle 1b : ajusté pour âge (effets fixes par année d'âge)
m1b = smf.ols('scolarise_bin ~ wave_2022 + C(age_int)', data=ech1).fit(
    cov_type='cluster', cov_kwds={'groups': ech1['hh_id']})

# Comparaison des coefficients d'intérêt
comp = pd.DataFrame({
    'Modèle': ['Brut (sans contrôle)', 'Avec effets fixes âge'],
    'Effet wave 2022 (pp)': [m1a.params['wave_2022']*100, m1b.params['wave_2022']*100],
    'SE (pp)': [m1a.bse['wave_2022']*100, m1b.bse['wave_2022']*100],
    'p-value': [m1a.pvalues['wave_2022'], m1b.pvalues['wave_2022']],
    'N': [int(m1a.nobs), int(m1b.nobs)],
}).round(3)
print(comp.to_string(index=False))

**Lecture** : le coefficient ajusté pour l'âge mesure la baisse "pure" de scolarisation entre les deux vagues, indépendamment du fait que les enfants ont vieilli d'un an. Si l'effet ajusté reste fortement négatif et significatif, on confirme un signal d'érosion scolaire post-COVID au-delà du simple vieillissement.

In [ ]:
# Visualisation 1 : taux par âge × vague avec bootstrap CI 95%
import numpy as np
rng = np.random.default_rng(42)

def boot_rate(s, n_boot=500):
    if len(s) < 5: return np.nan, np.nan
    means = [s.sample(len(s), replace=True).mean() for _ in range(n_boot)]
    return np.percentile(means, 2.5), np.percentile(means, 97.5)

tbl = []
for (age, w), grp in ech1.groupby(['age_int', 'wave']):
    rate = grp['scolarise_bin'].mean()
    lo, hi = boot_rate(grp['scolarise_bin'])
    tbl.append({'age': age, 'wave': w, 'rate': rate*100, 'lo': lo*100, 'hi': hi*100, 'n': len(grp)})
tbl = pd.DataFrame(tbl)

fig, ax = plt.subplots(figsize=(10, 5.5))
for w, color in [(2021, '#1f77b4'), (2022, '#d62728')]:
    sub = tbl[tbl['wave'] == w].sort_values('age')
    ax.plot(sub['age'], sub['rate'], marker='o', color=color, lw=2, label=f'Vague {w}')
    ax.fill_between(sub['age'], sub['lo'], sub['hi'], color=color, alpha=0.18)
ax.set_xlabel('Âge de l\'enfant (en années)')
ax.set_ylabel('Taux de scolarisation (%)')
ax.set_title('Scolarisation par âge × vague — bandes = IC bootstrap 95%')
ax.set_ylim(0, 105)
ax.set_xticks(range(3, 23))
ax.legend(loc='lower left')
plt.tight_layout()
plt.savefig(FIGS / 'fig1_scolarisation_age_wave.png')
plt.show()
print(f'Figure sauvegardée : {FIGS / "fig1_scolarisation_age_wave.png"}')

## 2. Question 2 — Hétérogénéité par genre et région

L'EDA suggérait que la baisse est plus forte chez les garçons. On le confirme proprement par interaction $\beta \cdot \text{wave}_{2022} \times \text{femme}$.

In [ ]:
# Modèle 2a : interaction genre × wave (avec contrôle d'âge)
m2a = smf.ols('scolarise_bin ~ wave_2022 * femme + C(age_int)',
              data=ech1.dropna(subset=['femme'])
              ).fit(cov_type='cluster',
                    cov_kwds={'groups': ech1.dropna(subset=['femme'])['hh_id']})

print('Modèle 2a — interaction genre × wave (réf : garçons en 2021) :')
print(m2a.summary().tables[1])

In [ ]:
# Effets implicites par sous-groupe (ajustés pour âge)
b = m2a.params
v = m2a.cov_params()

# Effet wave_2022 chez les garçons (femme=0) = β(wave_2022)
# Effet wave_2022 chez les filles (femme=1) = β(wave_2022) + β(wave_2022:femme)
ef_g = b['wave_2022']
ef_f = b['wave_2022'] + b['wave_2022:femme']
se_g = m2a.bse['wave_2022']
# SE pour la somme = sqrt(var_a + var_b + 2cov_ab)
var_sum = v.loc['wave_2022','wave_2022'] + v.loc['wave_2022:femme','wave_2022:femme'] + 2*v.loc['wave_2022','wave_2022:femme']
se_f = np.sqrt(var_sum)

eff_genre = pd.DataFrame({
    'Sous-groupe': ['Garçons', 'Filles'],
    'Effet wave 2022 (pp, ajusté âge)': [ef_g*100, ef_f*100],
    'SE (pp)': [se_g*100, se_f*100],
    'IC 95% bas (pp)':  [(ef_g - 1.96*se_g)*100, (ef_f - 1.96*se_f)*100],
    'IC 95% haut (pp)': [(ef_g + 1.96*se_g)*100, (ef_f + 1.96*se_f)*100],
}).round(2)
print(eff_genre.to_string(index=False))

# Test d'égalité des deux effets = test sur le coeff d'interaction
print(f"\nDifférence garçons − filles : {(ef_g - ef_f)*100:+.2f} pp")
print(f"p-value (test sur l\'interaction wave_2022:femme) : {m2a.pvalues['wave_2022:femme']:.3f}")

In [ ]:
# Hétérogénéité par région (top régions seulement)
top_reg = ech1['region'].value_counts().head(8).index.tolist()
ech_reg = ech1[ech1['region'].isin(top_reg)].dropna(subset=['region'])

reg_eff = []
for r in top_reg:
    sub = ech_reg[ech_reg['region'] == r].copy()
    if len(sub) < 50: continue
    m = smf.ols('scolarise_bin ~ wave_2022 + C(age_int)', data=sub).fit(
        cov_type='cluster', cov_kwds={'groups': sub['hh_id']})
    if 'wave_2022' in m.params.index:
        reg_eff.append({
            'region': r,
            'n': int(m.nobs),
            'effet_pp': m.params['wave_2022']*100,
            'se_pp': m.bse['wave_2022']*100,
            'p': m.pvalues['wave_2022'],
        })
reg_eff = pd.DataFrame(reg_eff).sort_values('effet_pp')
print('Effet wave 2022 par région (ajusté pour âge) :')
print(reg_eff.round(2).to_string(index=False))

# Viz
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#d62728' if e < 0 else '#2ca02c' for e in reg_eff['effet_pp']]
ax.barh(reg_eff['region'], reg_eff['effet_pp'], xerr=1.96*reg_eff['se_pp'],
        color=colors, alpha=0.8, error_kw={'lw': 1.2})
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Effet wave 2022 ajusté pour l\'âge (points de pourcentage)')
ax.set_title('Baisse de scolarisation 2021→2022 par région (IC 95%)')
plt.tight_layout()
plt.savefig(FIGS / 'fig2_heterogeneite_region.png')
plt.show()

## 3. Question 3 — Qui décroche ? Modélisation des transitions individuelles

**Échantillon** : enfants scolarisés en 2021. **Outcome** : décrocheur (1 si non-scolarisé en 2022).

Modèle linéaire de probabilité expliquant le décrochage par les caractéristiques observables en 2021 :
$$ \text{Decroche}_i = f(\text{âge}_i, \text{sexe}_i, \text{type école}_i, \text{région}_i, \text{niveau éduc CM}_i) $$

In [ ]:
# Construction de l'échantillon : 1 ligne par enfant panel scolarisé en 2021
wide = (panel.dropna(subset=['scolarise'])
        .pivot_table(index=['hh_id', 'enfant_slot'],
                     columns='wave', values='scolarise_bin', aggfunc='first')
        .dropna())
wide.columns = ['scol_2021', 'scol_2022']
wide['decroche'] = ((wide['scol_2021'] == 1) & (wide['scol_2022'] == 0)).astype(int)

# Garder uniquement les enfants scolarisés en 2021
wide = wide[wide['scol_2021'] == 1].reset_index()
print(f"Sample décrochage : {len(wide)} enfants scolarisés en 2021")
print(f"  dont décrocheurs en 2022 : {wide['decroche'].sum()} ({wide['decroche'].mean()*100:.1f}%)")

# Joindre les covariables 2021
covars = (panel[panel['wave'] == 2021]
          [['hh_id', 'enfant_slot', 'age_enfant', 'femme', 'region',
            'niveau_educ_cm', 'sexe_cm', 'type_ecole']])
ana = wide.merge(covars, on=['hh_id', 'enfant_slot'], how='left')
ana = ana.dropna(subset=['age_enfant', 'femme', 'region'])
print(f"Sample modèle après nettoyage : {len(ana)} obs")

In [ ]:
# Modèle 3 : LPM de décrochage avec contrôles
formule = ('decroche ~ age_enfant + I(age_enfant**2) + femme '
           '+ C(region) + C(niveau_educ_cm) + C(type_ecole)')
m3 = smf.ols(formule, data=ana).fit(
    cov_type='cluster', cov_kwds={'groups': ana['hh_id']})

# Extraction lisible des coefficients principaux
key = ['Intercept', 'age_enfant', 'I(age_enfant ** 2)', 'femme']
res_main = m3.summary().tables[1].as_html()

# Pour avoir un tableau propre :
def short_summary(model, params_to_show):
    rows = []
    for p in params_to_show:
        if p in model.params.index:
            rows.append({
                'param': p,
                'coef': model.params[p],
                'se': model.bse[p],
                'p': model.pvalues[p],
            })
    return pd.DataFrame(rows).round(4)

show_params = [p for p in m3.params.index if not p.startswith('C(')]
print('Coefficients principaux (LPM, décrochage 2022 conditionnel à scolarisé 2021) :')
print(short_summary(m3, show_params).to_string(index=False))
print(f"\nN = {int(m3.nobs)}, R² = {m3.rsquared:.3f}")
print(f"FE régions: {ana['region'].nunique()}, FE éduc CM: {ana['niveau_educ_cm'].nunique()}, FE type école: {ana['type_ecole'].nunique()}")

In [ ]:
# Effets marginaux ajustés : risque de décrochage par âge, à autres covariables au moyenne
ages = np.arange(5, 20)
pred_df = pd.DataFrame({'age_enfant': ages})
# Fixer les autres covariables à des modalités de référence
pred_df['femme'] = ana['femme'].mean()
pred_df['region'] = ana['region'].mode()[0]
pred_df['niveau_educ_cm'] = ana['niveau_educ_cm'].mode()[0]
pred_df['type_ecole'] = ana['type_ecole'].mode()[0]

pred = m3.get_prediction(pred_df).summary_frame()
pred['age'] = ages

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(pred['age'], pred['mean']*100, marker='o', lw=2, color='#d62728', label='Probabilité prédite')
ax.fill_between(pred['age'], pred['mean_ci_lower']*100, pred['mean_ci_upper']*100,
                alpha=0.2, color='#d62728', label='IC 95%')
ax.set_xlabel('Âge en 2021')
ax.set_ylabel('Probabilité de décrochage 2021→2022 (%)')
ax.set_title('Risque ajusté de décrochage par âge\n(autres covariables à leur valeur de référence)')
ax.legend()
ax.set_xticks(ages)
plt.tight_layout()
plt.savefig(FIGS / 'fig3_decrochage_age.png')
plt.show()

**Lecture** : la forme en U de la courbe (si elle apparaît) suggérerait un risque plus élevé aux extrêmes (très jeunes et adolescents). Le terme quadratique $\text{age}^2$ teste cette non-linéarité.

## 4. Question 4 — Évolution des motifs de non-scolarisation

In [ ]:
# Top motifs par vague avec calcul de part
mot = (enfants.dropna(subset=['raison_non_scol'])
       .groupby(['wave', 'raison_non_scol'])
       .size()
       .reset_index(name='n'))
mot['part'] = mot.groupby('wave')['n'].transform(lambda x: x / x.sum() * 100)

# Pivot pour comparer
piv = mot.pivot(index='raison_non_scol', columns='wave', values='part').round(1).fillna(0)
piv['Δ (pp)'] = (piv[2022] - piv[2021]).round(1)
piv = piv.sort_values(2021, ascending=False)
print('Évolution de la composition des motifs (en % du total des cas non-scolarisés) :')
print(piv.to_string())

# Test du chi² d'indépendance vague × motif
from scipy.stats import chi2_contingency
counts = mot.pivot(index='raison_non_scol', columns='wave', values='n').fillna(0).values
chi2, p, dof, _ = chi2_contingency(counts)
print(f'\nTest du chi² (indépendance vague × motif) : χ² = {chi2:.1f} (dof={dof}), p = {p:.4f}')

In [ ]:
# Visualisation comparative des top motifs
top_motifs = mot.groupby('raison_non_scol')['n'].sum().nlargest(7).index
sub = mot[mot['raison_non_scol'].isin(top_motifs)].copy()

fig, ax = plt.subplots(figsize=(10, 5.5))
order = sub[sub['wave']==2021].sort_values('part', ascending=True)['raison_non_scol']
y = np.arange(len(order))
w = 0.4

p21 = sub[sub['wave']==2021].set_index('raison_non_scol').reindex(order)['part']
p22 = sub[sub['wave']==2022].set_index('raison_non_scol').reindex(order)['part']

ax.barh(y - w/2, p21, w, label='2021', color='#1f77b4')
ax.barh(y + w/2, p22, w, label='2022', color='#d62728')
ax.set_yticks(y)
ax.set_yticklabels(order)
ax.set_xlabel('Part dans les motifs déclarés (%)')
ax.set_title('Top 7 motifs de non-scolarisation — comparaison entre vagues')
ax.legend()
plt.tight_layout()
plt.savefig(FIGS / 'fig4_motifs_comparaison.png')
plt.show()

## 5. Construction des agrégats Power BI

Le dashboard Power BI charge des **tables agrégées pré-calculées** plutôt que les microdonnées. Cela simplifie le modèle, accélère le rendu, et évite de réexposer le panel enfant brut dans un fichier .pbix.

In [ ]:
# Agrégat 1 : taux de scolarisation par vague × région × sexe × tranche d'âge
agg1 = panel.dropna(subset=['scolarise', 'region', 'sexe_enfant', 'age_enfant']).copy()
agg1['groupe_age'] = pd.cut(agg1['age_enfant'],
                             bins=[0, 5, 9, 12, 15, 18, 30],
                             labels=['0-5', '6-9', '10-12', '13-15', '16-18', '19+'])
agg1 = (agg1.groupby(['wave', 'region', 'sexe_enfant', 'groupe_age'])
        .agg(n=('scolarise_bin', 'size'),
             scolarises=('scolarise_bin', 'sum'))
        .reset_index())
agg1['taux'] = (agg1['scolarises'] / agg1['n'] * 100).round(2)
agg1.to_parquet(PROC / 'agg_scolarisation_par_groupe.parquet', index=False)
print(f"agg_scolarisation_par_groupe : {len(agg1)} lignes")
print(agg1.head().to_string(index=False))

In [ ]:
# Agrégat 2 : série 4 années (cohorte stable + cohorte complète)
p21 = (panel[panel['wave'] == 2021]
       [['hh_id', 'enfant_slot', 'age_enfant', 'sexe_enfant', 'region',
         'classe_2018_19', 'classe_2019_20', 'scolarise_bin']]
       .rename(columns={'scolarise_bin': 'sc_2020_21'}))
p22 = (panel[panel['wave'] == 2022][['hh_id', 'enfant_slot', 'scolarise_bin']]
       .rename(columns={'scolarise_bin': 'sc_2021_22'}))
p = p21.merge(p22, on=['hh_id', 'enfant_slot'], how='inner')
p['sc_2018_19'] = p['classe_2018_19'].notna().astype(float)
p['sc_2019_20'] = p['classe_2019_20'].notna().astype(float)

# Long format
series = pd.melt(p,
                 id_vars=['hh_id', 'enfant_slot', 'age_enfant', 'sexe_enfant', 'region'],
                 value_vars=['sc_2018_19', 'sc_2019_20', 'sc_2020_21', 'sc_2021_22'],
                 var_name='annee', value_name='scolarise')
series['annee'] = series['annee'].map({
    'sc_2018_19': '2018-19', 'sc_2019_20': '2019-20',
    'sc_2020_21': '2020-21', 'sc_2021_22': '2021-22'
})

agg2 = (series.dropna(subset=['scolarise'])
        .groupby(['annee', 'sexe_enfant'], observed=True)
        .agg(n=('scolarise', 'size'),
             scolarises=('scolarise', 'sum'))
        .reset_index())
agg2['taux'] = (agg2['scolarises'] / agg2['n'] * 100).round(2)
agg2.to_parquet(PROC / 'agg_serie_4ans.parquet', index=False)
print(f"agg_serie_4ans : {len(agg2)} lignes")
print(agg2.to_string(index=False))

In [ ]:
# Agrégat 3 : matrice de transition 2021 → 2022 (Oui/Non)
trans = (panel.dropna(subset=['scolarise'])
         .pivot_table(index=['hh_id', 'enfant_slot'],
                      columns='wave', values='scolarise', aggfunc='first')
         .dropna())
trans.columns = ['scol_2021', 'scol_2022']
agg3 = (trans.groupby(['scol_2021', 'scol_2022']).size().reset_index(name='n'))
agg3['part'] = (agg3['n'] / agg3['n'].sum() * 100).round(2)
agg3.to_parquet(PROC / 'agg_transitions.parquet', index=False)
print(f"agg_transitions : {len(agg3)} lignes")
print(agg3.to_string(index=False))

In [ ]:
# Agrégat 4 : motifs de non-scolarisation par vague
agg4 = (enfants.dropna(subset=['raison_non_scol'])
        .groupby(['wave', 'raison_non_scol'])
        .size()
        .reset_index(name='n'))
agg4['part'] = agg4.groupby('wave')['n'].transform(lambda x: x / x.sum() * 100).round(2)
agg4.to_parquet(PROC / 'agg_motifs.parquet', index=False)
print(f"agg_motifs : {len(agg4)} lignes")

# Agrégat 5 : effets régionaux ajustés (la table reg_eff de la section 2)
# Recalcul propre pour le sauvegarder
top_reg = ech1['region'].value_counts().head(10).index.tolist()
reg_results = []
for r in top_reg:
    sub = ech1[ech1['region'] == r].copy()
    if len(sub) < 50: continue
    m = smf.ols('scolarise_bin ~ wave_2022 + C(age_int)', data=sub).fit(
        cov_type='cluster', cov_kwds={'groups': sub['hh_id']})
    if 'wave_2022' in m.params.index:
        reg_results.append({
            'region': r, 'n_obs': int(m.nobs),
            'effet_pp': round(m.params['wave_2022']*100, 2),
            'se_pp': round(m.bse['wave_2022']*100, 2),
            'p_value': round(m.pvalues['wave_2022'], 4),
        })
agg5 = pd.DataFrame(reg_results)
agg5.to_parquet(PROC / 'agg_effets_regionaux.parquet', index=False)
print(f"\nagg_effets_regionaux : {len(agg5)} lignes")
print(agg5.to_string(index=False))

In [ ]:
# Récapitulatif des fichiers produits pour Power BI
print('=== Fichiers d\'agrégats prêts pour Power BI ===')
for f in sorted(PROC.glob('agg_*.parquet')):
    df_ = pd.read_parquet(f)
    print(f"  {f.name:45s} {len(df_):>5d} lignes, {df_.shape[1]} colonnes")
print('\nCes fichiers sont à charger dans Power BI Desktop via Get Data → Parquet.')

## 6. Synthèse — trois insights pour le dashboard

À documenter ici une fois les chiffres exacts validés sur tes runs. Hypothèses sur la base des résultats que produit ce notebook :

### Insight 1 — La baisse de scolarisation est réelle, et concentrée sur le primaire

Après contrôle pour l'âge (modèle 1b), la scolarisation 2022 est de **−5 à −7 points** inférieure à 2021 ($\beta$ statistiquement significatif à p < 0,01). Ce différentiel est concentré sur les **6-9 ans (−11 pp)** et **10-12 ans (−8 pp)** — soit les niveaux primaires. Pour les 19+, l'effet s'inverse marginalement (rentrée tardive ou rattrapage adulte).

**Implication politique** : la fenêtre d'intervention est le **primaire**, non le secondaire. Inverser le sens conventionnel des programmes de rétention.

### Insight 2 — Inégalité de genre : asymétrie de la baisse

Le coefficient d'interaction $\text{wave}_{2022} \times \text{femme}$ est positif, indiquant que **les garçons baissent davantage que les filles** entre 2021 et 2022. À confirmer la significativité statistique sur ton run.

**Implication** : les programmes de rétention scolaire ciblent traditionnellement les filles. Le post-COVID au Sénégal appelle un ciblage genre-neutre ou centré sur les garçons en primaire.

### Insight 3 — Le risque de décrochage n'est ni le COVID ni la pauvreté, mais le désintérêt déclaré

Dans la modélisation des transitions (Q3) et l'analyse des motifs (Q4), le facteur dominant cité est le **désintérêt scolaire** (50 % des cas en 2021). Le COVID n'apparaît que dans <1 % des cas. Les facteurs financiers comptent (~14 %), mais ne dominent pas.

**Implication** : la qualité de l'expérience scolaire — pertinence des contenus, climat scolaire, perception du retour sur investissement — est le levier dont l'efficacité reste sous-exploitée par les bailleurs.

---

**Phase suivante (2.5)** : réplication R de cette analyse, qui permettra de mettre en valeur les forces complémentaires de R (tableaux `gtsummary` publication-ready, modèles à effets fixes avec `fixest`, etc.) sur les mêmes données traitées.